## Load the required packages

In [1]:
import anndata
import scanpy as sc
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import os
import numpy as np
from pathlib import Path

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/scMoGNN"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")  
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")   
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]
train_data, test_data

(View of AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 View of AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Perform clustering on reference gene expression

In [5]:
train_data.layers["counts"] = train_data.X.copy()
train_data.X = train_data.X.astype(float)
sc.pp.normalize_per_cell(train_data, counts_per_cell_after=1e6)
sc.pp.log1p(train_data)
sc.pp.pca(train_data, n_comps=15)
sc.pp.neighbors(train_data)
sc.tl.leiden(train_data, key_added='groups', resolution=0.5)

## Perform clustering on query gene expression

In [6]:
test_data.layers["counts"] = test_data.X.copy()
test_data.X = test_data.X.astype(float)
sc.pp.normalize_per_cell(test_data, counts_per_cell_after=1e6)
sc.pp.log1p(test_data)
sc.pp.pca(test_data, n_comps=15)
sc.pp.neighbors(test_data)
sc.tl.leiden(test_data, key_added='groups', resolution=0.5)

## Save the clustering results and the raw gene expression matrix

In [7]:
mmwrite(f"{base_dir}/gene_expression_train.mtx", csr_matrix(train_data.layers["counts"]))
train_data.obs.to_csv(f"{base_dir}/cell_metadata_train.txt")
train_data.var.to_csv(f"{base_dir}/var_metadata_train.txt")

mmwrite(f"{base_dir}/gene_expression_test.mtx", csr_matrix(test_data.layers["counts"]))
test_data.obs.to_csv(f"{base_dir}/cell_metadata_test.txt")
test_data.var.to_csv(f"{base_dir}/var_metadata_test.txt")